# Atividade Prática 4: AutoML versus Meta-aprendizagem
**Disciplina:** IN1097 – Tópicos Avançados em Agentes Inteligentes 2 (2026.2)  
**Instituição:** Centro de Informática (CIn) – Universidade Federal de Pernambuco (UFPE)  
**Aluno:** Gabriel Victor  
**Domínio:** Avaliação de Qualidade de Imagem sem Referência (Blind/No-Reference IQA)  
**Repositório:** https://github.com/gabrielvmdvital/metalearning-iqa

---
### Objetivo e Continuidade Metodológica
Esta prática executa o sistema de AutoML **FLAML** (*Fast and Lightweight AutoML*, Wang et al., MLSys 2021) sobre os **56 datasets de IQA** (15.160 imagens, 9 atributos NSS: `brilho`, `contraste`, `snr`, `entropia`, `assimetria`, `curtose`, `colorfulness`, `spatial_frequency`, `blur_extent` e alvo MOS), comparando-o sob o **protocolo 5-Fold CV com `StandardScaler` intra-fold + LODO** e a **métrica de perda $L = 1 - \text{SRCC}$** com as soluções baseadas em meta-aprendizagem.

**Meta-características empregadas nos meta-modelos:**
- **$X_{\text{base}}$ (15 atributos, usada em M2 `HARRIS`):** `NumberOfInstances`, `NumberOfFeatures`, `NumberOfClasses`, `MajorityClassSize`, `MinorityClassSize`, `MajorityClassPercentage`, `MinorityClassPercentage`, `MeanMeansOfNumericAtts`, `MeanStdDevOfNumericAtts`, `MeanSkewnessOfNumericAtts`, `MeanKurtosisOfNumericAtts`, `MeanAttributeEntropy`, `NoiseToSignalRatio`, `IQA_MeanBrightness`, `IQA_MeanContrast`.
- **$X_{\text{ext}}$ (23 atributos = $X_{\text{base}} + 8$, usada em M3 `Warm-Start HPO`):** acrescenta `IQA_MeanColorfulness`, `IQA_MeanSpatialFreq`, `IQA_MeanBlurExtent`, `TargetCoV`, `MeanCorrWithTarget`, `MaxCorrWithTarget`, `MeanFeatureCorr`, `LinearRegressionMAE`.


In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display

base_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
proc_dir = base_dir / 'data' / 'processed'
nb_dir = base_dir / 'notebooks'
print("Base dir:", base_dir)


Base dir: C:\Users\Victor\Documents\UFPE\MESTRADO\metalearning\metalearning-iqa


## 1. Etapa 1: Caracterização do FLAML e Verificação de Sobreposição

1. **Espaço de Busca:** $\chi = (l, h, s, r)$ com *learners* `lgbm`, `rf`, `xgboost`, `extra_tree`, `xgb_limitdepth` e `sgd`; hiperparâmetros orientados a custo e tamanho da amostra $s$.
2. **Estratégia de Busca:** Alocação de *learners* por $\text{ECI}(l)$ (*Estimated Cost for Improvement* / *FairChance*) + busca local frugal CFO (*Cost-Frugal Optimization*) partindo da configuração mais barata (`n_estimators=4`, `max_leaves=4`).
3. **Meta-conhecimento e Sobreposição:** `starting_points='data'` utiliza 22 configurações de referência extraídas do **OpenML** (`flaml/default/all/regression.json`). **Sobreposição zero** com nossos 56 datasets de IQA (TID2013, CSIQ, KADID-10k, ChallengeDB). Executamos também `starting_points='static'` para medir a contribuição do meta-conhecimento interno.
4. **Modelo Final e Orçamento:** Modelo único por padrão (`ensemble=False`) e ablação com `StackingRegressor` (`ensemble=True`). Controle via `time_budget` por fold (`0.3s`, `1.0s`, `3.0s` $\rightarrow$ `1.5s`, `5.0s`, `15.0s` por dataset) com `n_jobs=1`.


In [2]:
import flaml.default, os
reg_meta = json.load(open(os.path.join(os.path.dirname(flaml.default.__file__), 'all', 'regression.json')))
print("Meta-features do FLAML:", reg_meta['meta_feature_names'])
print("Nº de datasets/configurações no portfólio OpenML do FLAML:", len(reg_meta['neighbors']))
print("Fontes OpenML do portfólio:", reg_meta['configsource'])


Meta-features do FLAML: ['NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses', 'PercentageOfNumericFeatures']
Nº de datasets/configurações no portfólio OpenML do FLAML: 22
Fontes OpenML do portfólio: ['lgbm/houses', 'lgbm/house_8L', 'lgbm/poker', 'lgbm/default', 'xgboost/Albert', 'xgboost/mv', 'xgboost/bng_echomonths', 'xgboost/house_16H', 'xgb_limitdepth/higgs', 'xgb_limitdepth/bng_pharynx', 'xgb_limitdepth/connect-4', 'xgb_limitdepth/house_16H', 'xgb_limitdepth/bng_echomonths', 'xgb_limitdepth/default', 'rf/houses', 'rf/poker', 'rf/bank-marketing', 'rf/default', 'extra_tree/house_16H', 'extra_tree/default', 'extra_tree/dilbert', 'extra_tree/particulate-matter']


## 2. Etapa 2: Mesmo Protocolo de Fato: Matriz $P_{\text{tabular}}$ e Abordagens P1/P2 em LODO


In [3]:
df_p_tab = pd.read_csv(proc_dir / 'P_matrix_tabular.csv', index_col=0)
df_l_tab = pd.read_csv(proc_dir / 'L_matrix_tabular.csv', index_col=0)
df_t_tab = pd.read_csv(proc_dir / 'P_matrix_tabular_times.csv', index_col=0)

p_summary = pd.DataFrame({
    'SRCC Médio': df_p_tab.mean(),
    'Perda L Média': df_l_tab.mean(),
    'Tempo Médio/DS (s)': df_t_tab.mean()
}).sort_values('Perda L Média')
print("Portfólio P_tabular (6 regressores default nos 5 mesmos folds):")
display(p_summary.round(4))

df_p1p2 = pd.read_csv(proc_dir / 'p4_metalearning_p1p2_tabular.csv')
print("\nDesempenho das abordagens das Práticas 1 e 2 sobre P_tabular (LODO):")
display(df_p1p2[['approach', 'mean_top1_loss', 'mean_top1_srcc', 'mean_ranking_srcc', 'mean_loss_curve_auc']].head(10).round(4))


Portfólio P_tabular (6 regressores default nos 5 mesmos folds):
              SRCC Médio  Perda L Média  Tempo Médio/DS (s)
ExtraTrees        0.6631         0.3369              0.7982
RandomForest      0.5807         0.4193              1.1407
LightGBM          0.4984         0.5016              0.1283
SVR               0.4039         0.5961              0.0356
MLP               0.3709         0.6291              1.1444
kNN               0.2558         0.7442              0.0132

Desempenho das abordagens das Práticas 1 e 2 sobre P_tabular (LODO):
                        approach  mean_top1_loss  mean_top1_srcc  mean_ranking_srcc  mean_loss_curve_auc
0                             AR          0.3369          0.6631             0.7976               0.0121
1                             MR          0.3369          0.6631             0.7976               0.0121
2       HARRIS (L=0.25) (X_base)          0.3396          0.6604             0.7693               0.0124
3   HARRIS (L=0.0) (X_este

## 3. Análise 1: Desempenho, Regret e Testes de Friedman + Nemenyi


In [4]:
df_summary = pd.read_csv(proc_dir / 'p4_experiment_summary.csv')
print("Resumo Geral (Desempenho, Regret e Custo Online):")
display(df_summary.round(4))

df_fried = pd.read_csv(proc_dir / 'p4_friedman_tests.csv')
print("\nTestes de Friedman e Wilcoxon por Orçamento do FLAML:")
display(df_fried)

display(Image(filename=str(nb_dir / 'fig_p4_1_desempenho_boxplots.png')))
display(Image(filename=str(nb_dir / 'fig_p4_2_cd_diagrams_orcamentos.png')))


Resumo Geral (Desempenho, Regret e Custo Online):
                       Método  SRCC_Medio  Perda_L_Media  Perda_L_Desv  Regret_MelhorObs  Regret_OraculoGlobal  Custo_Online_Modelo_s  Custo_Online_Total_s
0       M1: Melhor Médio (AR)      0.6631         0.3369        0.2423            0.0763                0.0865                 0.7982                4.6219
1    M2: Meta-Learning (LODO)      0.6604         0.3396        0.2421            0.0790                0.0891                 0.8093                4.6330
2    M3: Warm-Start HPO (T=5)      0.7222         0.2778        0.2384            0.0172                0.0274                 0.5983                4.4219
3   M3: Warm-Start HPO (T=20)      0.7375         0.2625        0.2265            0.0019                0.0121                 7.3290               11.1527
4          FLAML Curto (1.5s)      0.3779         0.6221        0.2660            0.3615                0.3717                 1.7047                5.5284
5          FLA

## 4. Análise 2: Desempenho vs. Custo (Online vs. Offline e Break-Even)


In [5]:
with open(proc_dir / 'p4_offline_costs.json', 'r', encoding='utf-8') as f:
    offline = json.load(f)
print("Custos Offline Medidos (55 datasets de treino):")
for k, v in offline.items():
    print(f"  {k}: {v:.2f}s")

print("\nBreak-Even (Nº de novos datasets para amortizar o custo offline):")
for m_lbl, c_off_key in [
    ('M2: Meta-Learning (LODO)', 'offline_m2_total_s'),
    ('M3: Warm-Start HPO (T=5)', 'offline_m3_total_s'),
    ('M3: Warm-Start HPO (T=20)', 'offline_m3_total_s'),
]:
    c_on_m = float(df_summary.loc[df_summary['Método'] == m_lbl, 'Custo_Online_Total_s'].iloc[0])
    c_off = offline[c_off_key]
    for a_lbl in ['FLAML Curto (1.5s)', 'FLAML Médio (5.0s)', 'FLAML Longo (15.0s)']:
        c_on_a = float(df_summary.loc[df_summary['Método'] == a_lbl, 'Custo_Online_Total_s'].iloc[0])
        diff = c_on_a - c_on_m
        n_be = int(np.ceil(c_off / diff)) if diff > 0 else 'Nunca (online >= AutoML)'
        print(f"  {m_lbl:28s} vs {a_lbl:20s}: N* = {n_be}")

display(Image(filename=str(nb_dir / 'fig_p4_3_desempenho_vs_custo.png')))


Custos Offline Medidos (55 datasets de treino):
  offline_feat_55_s: 210.30s
  offline_ptab_55_s: 179.32s
  offline_meta_fit_s: 0.91s
  offline_m2_total_s: 390.53s
  offline_hpo_matrix_55_s: 155.50s
  offline_m3_total_s: 365.80s

Break-Even (Nº de novos datasets para amortizar o custo offline):
  M2: Meta-Learning (LODO)     vs FLAML Curto (1.5s)  : N* = 437
  M2: Meta-Learning (LODO)     vs FLAML Médio (5.0s)  : N* = 82
  M2: Meta-Learning (LODO)     vs FLAML Longo (15.0s) : N* = 26
  M3: Warm-Start HPO (T=5)     vs FLAML Curto (1.5s)  : N* = 331
  M3: Warm-Start HPO (T=5)     vs FLAML Médio (5.0s)  : N* = 73
  M3: Warm-Start HPO (T=5)     vs FLAML Longo (15.0s) : N* = 24
  M3: Warm-Start HPO (T=20)    vs FLAML Curto (1.5s)  : N* = Nunca (online >= AutoML)
  M3: Warm-Start HPO (T=20)    vs FLAML Médio (5.0s)  : N* = Nunca (online >= AutoML)
  M3: Warm-Start HPO (T=20)    vs FLAML Longo (15.0s) : N* = 43
<IPython.core.display.Image object>


## 5. Análise 3: O que o AutoML Escolheu e Correlação com Meta-Conhecimento


In [6]:
df_choices = pd.read_csv(proc_dir / 'p4_automl_choices_analysis.csv')
print("Distribuição Modal de Algoritmos (FLAML Longo, 15s/DS):")
print(df_choices['flaml_longo_modal'].value_counts())
print(f"\nTaxa de pertencimento a P_tabular (FLAML Longo modal): {df_choices['flaml_longo_in_ptab'].mean()*100:.1f}%")
print(f"Taxa de concordância FLAML Longo modal == M2: {df_choices['flaml_longo_agrees_m2'].mean()*100:.1f}%")
print(f"Datasets onde FLAML Longo superou M2: {(df_choices['gain_flaml_longo_vs_m2'] > 0).sum()}/56")
print(f"Datasets onde FLAML Longo superou Warm-Start HPO (T=20): {(df_choices['gain_flaml_longo_vs_hpo20'] > 0).sum()}/56")

df_corrs = pd.read_csv(proc_dir / 'p4_correlations_meta.csv')
print("\nPrincipais Correlações (Spearman p < 0.05) entre Ganho do AutoML e Meta-Conhecimento:")
display(df_corrs[df_corrs['p_value'] < 0.05].sort_values('p_value').round(4))

display(Image(filename=str(nb_dir / 'fig_p4_4_escolhas_automl.png')))
display(Image(filename=str(nb_dir / 'fig_p4_5_ganho_vs_metaconhecimento.png')))


Distribuição Modal de Algoritmos (FLAML Longo, 15s/DS):
flaml_longo_modal
lgbm          27
sgd           16
extra_tree     9
rf             2
xgboost        2
Name: count, dtype: int64

Taxa de pertencimento a P_tabular (FLAML Longo modal): 67.9%
Taxa de concordância FLAML Longo modal == M2: 16.1%
Datasets onde FLAML Longo superou M2: 2/56
Datasets onde FLAML Longo superou Warm-Start HPO (T=20): 0/56

Principais Correlações (Spearman p < 0.05) entre Ganho do AutoML e Meta-Conhecimento:
                         Alvo             Variavel  Spearman_rho  p_value
7      gain_flaml_longo_vs_m2   IQA_MeanBlurExtent       -0.4857   0.0001
6      gain_flaml_longo_vs_m2  IQA_MeanSpatialFreq       -0.4765   0.0002
19  gain_flaml_longo_vs_hpo20   IQA_MeanBlurExtent       -0.4542   0.0004
18  gain_flaml_longo_vs_hpo20  IQA_MeanSpatialFreq       -0.4519   0.0005
20  gain_flaml_longo_vs_hpo20            TargetCoV        0.4310   0.0009
2      gain_flaml_longo_vs_m2    NumberOfInstances        0.3613 